In [1]:
import pandas as pd
import numpy as np

from validation_metrics import ValidationGate, VoltammogramFidelityIndex

# Data
raw_potential_grid = 'raw/raw_potential_grid.csv'
raw_signals_real = 'raw/raw_signals_real.csv'
raw_signals_augmented = 'raw/raw_signals_augmented.csv'
# raw_signals_combined = 'raw/raw_combined_signals.csv'

gate = ValidationGate.from_csv(raw_potential_grid, raw_signals_real)

results = gate.evaluate_csv(raw_signals_augmented, run_tiers=[1, 2])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 0.962  (≥0.90)                ║
║    JSD mean / max                     : 0.0749 / 0.2259           ║
║    MMD²                               : 0.019505                ║
║    SWD mean / max                     : 0.1495 / 0.3119           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 0.833  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9795  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.09473  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 0.8090  [Good     ]  ║
║  NEW  P

In [10]:
import pandas as pd
import numpy as np

from validation_metrics import ValidationGate, VoltammogramFidelityIndex

# Data
raw_potential_grid = 'raw/raw_potential_grid.csv'
raw_signals_real = 'raw/raw_signals_real.csv'
raw_signals_augmented = 'raw/timegan_sdv_signals.csv'
# raw_signals_combined = 'raw/raw_combined_signals.csv'

gate = ValidationGate.from_csv(raw_potential_grid, raw_signals_real)

results = gate.evaluate_csv(raw_signals_augmented, run_tiers=[1, 2])

vfi = VoltammogramFidelityIndex.from_gate_results(results, verbose=True)


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ❌ FAIL              ║
║    KS mean frac                       : 0.574  (≥0.90)                ║
║    JSD mean / max                     : 0.3193 / 0.8720           ║
║    MMD²                               : 0.150862                ║
║    SWD mean / max                     : 3.2343 / 22.5746           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ❌ FAIL              ║
║    PFF class pass frac                : 0.333  (≥0.70)                ║
║    Randles-Ševčík R²                  : -146.2156  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.96352  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 0.1793  [Poor     ]  ║
║  NE

In [11]:

gate = ValidationGate.from_csv(raw_potential_grid, raw_signals_real)

results_sanity = gate.evaluate_csv(raw_signals_real, run_tiers=[1, 2])

vfi_perfect = VoltammogramFidelityIndex.from_gate_results(results_sanity, verbose=True)
print(f"VFI Sanity Check: {vfi_perfect:.4f}") 


╔══════════════════════════════════════════════════════════╗
║           VALIDATION GATE - FINAL DECISION              ║
╠══════════════════════════════════════════════════════════╣
║  Tier 1 - Statistical Distributional  : ✅ PASS              ║
║    KS mean frac                       : 1.000  (≥0.90)                ║
║    JSD mean / max                     : 0.0000 / 0.0000           ║
║    MMD²                               : -0.024235                ║
║    SWD mean / max                     : 0.0000 / 0.0000           ║
╠══════════════════════════════════════════════════════════╣
║  Tier 2 - Physics-Anchored            : ✅ PASS              ║
║    PFF class pass frac                : 1.000  (≥0.70)                ║
║    Randles-Ševčík R²                  : 0.9775  (≥0.90)         ║
║    ACF Δ (full signal)                : 0.00000  (<0.10)        ║
╠══════════════════════════════════════════════════════════╣
║  NEW  VoltammogramFidelityIndex (VFI)  : 1.0000  [Excellent]  ║
║  NEW  

In [15]:
import pandas as pd
import numpy as np
from validation_metrics import quick_compare

df_real = pd.read_csv('raw/raw_signals_real.csv')
E = pd.read_csv('raw/raw_potential_grid.csv').values.flatten()

y_real = df_real['concentration'].values
X_real = df_real.drop(columns=['concentration']).values

files_to_test = {
    'Augmented_Baseline': 'raw/raw_signals_augmented.csv',
    'TimeGAN_SDV':        'raw/timegan_sdv_signals.csv'
}

batches = {}
for name, path in files_to_test.items():
    df_synth = pd.read_csv(path)
    y_synth = df_synth['concentration'].values
    X_synth = df_synth.drop(columns=['concentration']).values
    batches[name] = (X_synth, y_synth)


comparison_df = quick_compare(E, X_real, y_real, batches, run_tiers=[1, 2])


print(comparison_df.to_string())

                batch  KS_mean_frac  JSD_mean      MMD2  SWD_mean  PFF_class_frac  RS_R2_synth  delta_ACF     VFI VFI_label  PDF_overlap_mean  tier1_pass  tier2_pass
0  Augmented_Baseline        0.9624    0.0749  0.019505    0.1495           0.833       0.9795    0.09473  0.6702  Marginal            0.7072        True        True
1         TimeGAN_SDV        0.5739    0.3193  0.150862    3.2343           0.333    -146.2156    0.96352  0.1793      Poor            0.4553       False       False
